# TP2 — Le catalogue des livres
## Pipeline Data Complet : API Open Library → Python → Power BI
- Auteur : SELMAN Marjane
- Date : Août 2026
- Formation : B1 IA & Data — Ynov Campus Lyon
### Contexte
Data Analyst pour un groupe d'édition envisageant de racheter des librairies
indépendantes. Objectif : analyser la structure du marché du livre via l'API
Open Library pour orienter la décision du comité de direction.

### Vérification préalable de l'API
Avant l'extraction, test de l'API Open Library sur Postman pour valider sa
disponibilité et comprendre la structure de réponse JSON.
![Test Postman](test_postman.png)
Résultat : statut 200 OK confirmé, structure JSON conforme à la documentation.

In [14]:
# ==============================
# Etape 1 :  Extraction
# ==============================
# 1.1 - Exploration de l'API

In [15]:
## De 1.1.1 à 1.1.2

import requests 
import pandas as pd 
import time 

url = "https://openlibrary.org/search.json?q=fantasy&limit=5&page=1"
reponse = requests.get(url, timeout=30)
print(reponse.status_code)
data = reponse.json()
print(data)

200
{'numFound': 150861, 'start': 0, 'numFoundExact': True, 'num_found': 150861, 'documentation_url': 'https://openlibrary.org/dev/docs/api/search', 'q': 'fantasy', 'offset': None, 'docs': [{'author_key': ['OL666139A'], 'author_name': ['Carole Mortimer'], 'cover_edition_key': 'OL10738221M', 'cover_height': 3267, 'cover_i': 8337298, 'cover_width': 1833, 'ebook_access': 'printdisabled', 'edition_count': 6, 'first_publish_year': 1983, 'has_fulltext': True, 'ia': ['fantasygirl0000mort_s1b6', 'fantasygirl0000mort'], 'ia_collection': ['internetarchivebooks', 'printdisabled'], 'key': '/works/OL3775422W', 'language': ['und', 'eng'], 'public_scan_b': False, 'title': 'Fantasy Girl'}, {'author_key': ['OL661093A'], 'author_name': ['Penny Jordan'], 'cover_edition_key': 'OL10739155M', 'cover_height': 3264, 'cover_i': 6611725, 'cover_width': 1997, 'ebook_access': 'borrowable', 'edition_count': 4, 'first_publish_year': 1996, 'has_fulltext': True, 'ia': ['herchristmasfant0000jord', 'herchristmasfant00j

> *Rappel* : **"data"** corresponds à l'appel complet de l'API donc toutes les données

In [16]:
## 1.1.3
print(data['docs'][0].keys())

dict_keys(['author_key', 'author_name', 'cover_edition_key', 'cover_height', 'cover_i', 'cover_width', 'ebook_access', 'edition_count', 'first_publish_year', 'has_fulltext', 'ia', 'ia_collection', 'key', 'language', 'public_scan_b', 'title'])


In [17]:
## 1.1.4
livre = data['docs'][0]
print(livre.get('title'))
print(livre.get('author_name'))
print(livre.get('author_key'))
print(livre.get('first_publish_year'))
print(livre.get('edition_count'))
print(livre.get('number_of_pages_median'))
print(livre.get('language'))
print(livre.get('ebook_access'))
print(livre.get('subject'))

Fantasy Girl
['Carole Mortimer']
['OL666139A']
1983
6
None
['und', 'eng']
printdisabled
None


> *Rappel* : ".get" permet de renvoyer **None** si la clé n'existe pas sans planter 

> **Q.5** : les champs 'number_of_pages_median' et 'subject' sont absents car Open Library est une base collaborative avec des sources variée. Ainsi, le niveau de détail dépend de chaque contribution et des ressources disponibles. 

> *Ressources mobilisées durant cette partie avec le cours **Memo API Python*** : 
    1. import
    2. appel API
    3. vérification du statut de la requête
    4. voir les principales clés 

In [18]:
five_first_book = data['docs'][:5]

count_pages = 0
count_ebook = 0
for livre in five_first_book:
   if 'number_of_pages_median' in livre:
       count_pages += 1
   if 'ebook_access' in livre:
       count_ebook += 1
print(count_pages)
print(count_ebook)

0
5


> **Q1.A** :
Sur les 5 premiers livres : 0 ont `number_of_pages_median`, 5 ont `ebook_access`.
Cela montre que la complétude des données Open Library varie selon le champ. Certains sont systématiquement renseignés, d'autres non. D'où l'importance de
`.get()` pour éviter les erreurs.

In [19]:
# 1.2 - Collecte multi-sujets (8 sujets, 3 pages chacun)
sujets_livre = ['science fiction', 'roman historique', 'thriller', 'biographie', 'fantasy', 'philosophie', 'roman policier', 'developpement personnel']
print(len(sujets_livre)) #juste pr verifeir que bien 8 livres

8


In [20]:
for sujet in sujets_livre:
    print(sujet)

science fiction
roman historique
thriller
biographie
fantasy
philosophie
roman policier
developpement personnel


In [21]:
for sujet in sujets_livre:
    for page in range(1,4): # rappel : python s'arrete juste avant 4 pour avoir 1 a 3
        print(sujet, page)

science fiction 1
science fiction 2
science fiction 3
roman historique 1
roman historique 2
roman historique 3
thriller 1
thriller 2
thriller 3
biographie 1
biographie 2
biographie 3
fantasy 1
fantasy 2
fantasy 3
philosophie 1
philosophie 2
philosophie 3
roman policier 1
roman policier 2
roman policier 3
developpement personnel 1
developpement personnel 2
developpement personnel 3


In [22]:
url_base = "https://openlibrary.org/search.json"
for sujet in sujets_livre:
    for page in range(1,4): # rappel : python s'arrete juste avant 4 pour avoir 1 a 3
        reponse_page = requests.get(url_base, params={"q": sujet, "limit": 100, "page": page },
        timeout=30
        )
        time.sleep(1)
        print(sujet, page, reponse_page.status_code)

science fiction 1 200
science fiction 2 200
science fiction 3 200
roman historique 1 200
roman historique 2 200
roman historique 3 200
thriller 1 500
thriller 2 200
thriller 3 500
biographie 1 500
biographie 2 500
biographie 3 500
fantasy 1 200
fantasy 2 200
fantasy 3 200
philosophie 1 200
philosophie 2 200
philosophie 3 500
roman policier 1 500
roman policier 2 200
roman policier 3 200
developpement personnel 1 200
developpement personnel 2 200
developpement personnel 3 200


*Ressources mobilisées durant cette partie avec le cours **Memo API Python*** : 1. extraire liste, 2. 2eme appel API et pour for sur internet